# Notebook for evaluating the quality of the values from BASt

## Checking to which degree the data is usable for answering our research questions or if adjustments have to be made.

In [1]:
import polars as pl

In [30]:
# read data file
df = pl.read_csv("All_Data_for_Kiel.csv")

# "a" for missing value
# "d" for faulty value 
# "s" for estimated value by the system of the counting station because of one missing value in that hour
error_flags = ["a", "d", "s"]

#check the directions of travel and show how many errors occured per counting station of which type 
error_report = (
    df.filter(
        (pl.col("K_KFZ_R1").is_in(error_flags)) |
        (pl.col("K_KFZ_R2").is_in(error_flags))
    )
    .group_by("Zst", "Strklas", "Strnum")
    .agg(
        pl.len().alias("Anzahl_Fehler"),
        ((pl.col("K_KFZ_R1") == "a") | (pl.col("K_KFZ_R2") == "a")).sum().alias("Type a fault"),
        ((pl.col("K_KFZ_R1") == "d") | (pl.col("K_KFZ_R2") == "d")).sum().alias("Type d fault"),
        ((pl.col("K_KFZ_R1") == "s") | (pl.col("K_KFZ_R2") == "s")).sum().alias("Type s; estimated value")
        
    )
)
# count all rows where both travel directions got marked with the optimal sensor reading quality per counting station ("-")
correct_report = (
    df
    .group_by("Zst", "Strklas", "Strnum")
    .agg(
        ((pl.col("K_KFZ_R1") == "-") & (pl.col("K_KFZ_R2") == "-")).sum().alias("Correct reading")
    )
)

# combine both reports to get one visually pleasing table
final_report = (
    error_report
    .join(correct_report, on = ["Zst", "Strklas", "Strnum"], how = "left")
    .with_columns(
        (pl.col("Anzahl_Fehler") + pl.col("Correct reading")).alias("Gesamtanzahl")
    )
    # calculate the percentage of rows where at least one value got marked with "s" (being estimated)
    .with_columns(
        ((pl.col("Type s; estimated value") / pl.col("Gesamtanzahl")) * 100).round(2).alias("Schätz-Quote in %")
    )
)
final_report

Zst,Strklas,Strnum,Anzahl_Fehler,Type a fault,Type d fault,Type s; estimated value,Correct reading,Gesamtanzahl,Schätz-Quote in %
i64,str,str,u32,u32,u32,u32,u32,u32,f64
1135,"""B""",""" 76""",30687,2231,0,28456,0,30687,92.73
1162,"""A""",""" 210""",43814,43814,0,0,0,43814,0.0
1158,"""B""",""" 502""",27784,2231,0,25553,2857,30641,83.39
1104,"""A""",""" 215""",4164,0,0,4164,39560,43724,9.52
1194,"""A""",""" 215""",10321,0,0,10321,33396,43717,23.61
1116,"""B""",""" 76""",35006,2207,0,32799,4368,39374,83.3
1112,"""B""",""" 503""",15763,3671,0,12092,15919,31682,38.17
1111,"""B""",""" 503""",13589,3671,0,9918,15204,28793,34.45
1156,"""A""",""" 21""",4157,0,0,4157,39133,43290,9.6


In [28]:
# list of counting stations that got selected
counting_stations = [1116, 1111, 1112, 1158, 1135, 1156, 1104, 1162, 1194]

# count how many data entries are in the data set per counting station
counts_per_station = (
    df
    .filter(pl.col("Zst").is_in(counting_stations))
    .group_by("Zst")
    .agg(
        pl.len().alias("Gesamtanzahl")
    )
)
    
print(counts_per_station)

shape: (9, 2)
┌──────┬──────────────┐
│ Zst  ┆ Gesamtanzahl │
│ ---  ┆ ---          │
│ i64  ┆ u32          │
╞══════╪══════════════╡
│ 1112 ┆ 32136        │
│ 1194 ┆ 43824        │
│ 1158 ┆ 30696        │
│ 1111 ┆ 32856        │
│ 1162 ┆ 43824        │
│ 1156 ┆ 43824        │
│ 1135 ┆ 30696        │
│ 1116 ┆ 39432        │
│ 1104 ┆ 43824        │
└──────┴──────────────┘


In [27]:
# calculate how many errors in total occured for each counting station (pre-processing for the table 
# at the top of this notebook)

top_error_zst = (
    error_report
    .group_by("Zst", "Strklas", "Strnum")
    .agg(
        pl.col("Anzahl_Fehler").sum().alias("Gesamtfehler im Zeitraum")
    )
    .sort("Gesamtfehler im Zeitraum", descending = True)
)

print(top_error_zst)

shape: (9, 4)
┌──────┬─────────┬────────┬──────────────────────────┐
│ Zst  ┆ Strklas ┆ Strnum ┆ Gesamtfehler im Zeitraum │
│ ---  ┆ ---     ┆ ---    ┆ ---                      │
│ i64  ┆ str     ┆ str    ┆ u32                      │
╞══════╪═════════╪════════╪══════════════════════════╡
│ 1162 ┆ A       ┆  210   ┆ 43814                    │
│ 1116 ┆ B       ┆   76   ┆ 35006                    │
│ 1135 ┆ B       ┆   76   ┆ 30687                    │
│ 1158 ┆ B       ┆  502   ┆ 27784                    │
│ 1112 ┆ B       ┆  503   ┆ 15763                    │
│ 1111 ┆ B       ┆  503   ┆ 13589                    │
│ 1194 ┆ A       ┆  215   ┆ 10321                    │
│ 1104 ┆ A       ┆  215   ┆ 4164                     │
│ 1156 ┆ A       ┆   21   ┆ 4157                     │
└──────┴─────────┴────────┴──────────────────────────┘


In [8]:
# this was pre-processing for the table above

all_entries_zst_1162 = (
    df
    .filter(pl.col("Zst") == 1162)
    .group_by("Zst", "Strklas", "Strnum")
    .agg(
        pl.len().alias("Anzahl Einträge Zst 1162")
    )
)
print(all_entries_zst_1162)

shape: (1, 4)
┌──────┬─────────┬────────┬──────────────────────────┐
│ Zst  ┆ Strklas ┆ Strnum ┆ Anzahl Einträge Zst 1162 │
│ ---  ┆ ---     ┆ ---    ┆ ---                      │
│ i64  ┆ str     ┆ str    ┆ u32                      │
╞══════╪═════════╪════════╪══════════════════════════╡
│ 1162 ┆ A       ┆  210   ┆ 43824                    │
└──────┴─────────┴────────┴──────────────────────────┘


In [10]:
# checking for counting station 1162 if there are any entries with an actual sensor reading (not '0')
check_1162 = (
    df
    .filter((pl.col("Zst") == 1162) & 
            (pl.col("K_KFZ_R1") != "a") &
            (pl.col("K_KFZ_R2") != "a")
           )
    .select(["Zst", "Strklas", "Strnum", "Datum", "KFZ_R1", "KFZ_R2", "K_KFZ_R1", "K_KFZ_R2"])
)

print(check_1162)

shape: (10, 8)
┌──────┬─────────┬────────┬────────┬────────┬────────┬──────────┬──────────┐
│ Zst  ┆ Strklas ┆ Strnum ┆ Datum  ┆ KFZ_R1 ┆ KFZ_R2 ┆ K_KFZ_R1 ┆ K_KFZ_R2 │
│ ---  ┆ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---      ┆ ---      │
│ i64  ┆ str     ┆ str    ┆ i64    ┆ str    ┆ str    ┆ str      ┆ str      │
╞══════╪═════════╪════════╪════════╪════════╪════════╪══════════╪══════════╡
│ 1162 ┆ A       ┆  210   ┆ 210328 ┆     0  ┆     0  ┆ z        ┆ z        │
│ 1162 ┆ A       ┆  210   ┆ 211031 ┆     0  ┆     0  ┆ z        ┆ z        │
│ 1162 ┆ A       ┆  210   ┆ 220327 ┆     0  ┆     0  ┆ z        ┆ z        │
│ 1162 ┆ A       ┆  210   ┆ 221030 ┆     0  ┆     0  ┆ z        ┆ z        │
│ 1162 ┆ A       ┆  210   ┆ 230326 ┆     0  ┆     0  ┆ z        ┆ z        │
│ 1162 ┆ A       ┆  210   ┆ 231029 ┆     0  ┆     0  ┆ z        ┆ z        │
│ 1162 ┆ A       ┆  210   ┆ 240331 ┆     0  ┆     0  ┆ z        ┆ z        │
│ 1162 ┆ A       ┆  210   ┆ 241027 ┆     0  ┆     0  ┆ z     

All values in of the couting station 1162 are Zero. Thus we delete them from our set.